#### Importing libraries

In [1]:
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import calendar

#### Files

In [ ]:


HOUSE_FILES = {
    "1 RK": "appliances_1RK.csv",
    "1 BHK": "appliances_1BHK.csv",
    "1.5 BHK": "appliances_15BHK.csv",
    "2 BHK": "appliances_2BHK.csv",
    "2.5 BHK": "appliances_25BHK.csv",
    "3 BHK": "appliances_3BHK.csv",
}
DATA_DIR = "./" 

def load_house_data(house_type):
    df = pd.read_csv(DATA_DIR + HOUSE_FILES[house_type])
    df = df[df["count"] > 0].reset_index(drop=True)
    records = df.to_dict("records")
    for r in records:
        r["hours"] = r.pop("default_hours")
    return records

#### Cost + seasonal logic

In [2]:
LOCATION_SLABS = {
    "Maharashtra (Pune)":        [(100, 4.71), (300, 10.29), (500, 13.10), (float("inf"), 14.68)],
    "Delhi":                     [(200, 3.0), (400, 4.5), (800, 6.5), (float("inf"), 7.0)],
    "Karnataka (Bangalore)":     [(100, 4.15), (200, 5.6), (float("inf"), 7.15)],
    "Tamil Nadu (Chennai)":      [(100, 0.0), (200, 2.35), (500, 4.7), (float("inf"), 6.35)],
    "Custom Flat Rate (₹8/unit)": [(float("inf"), 8.0)],
}

def calculate_bill(units, slabs):
    if units <= 0:
        return 0.0
    remaining, prev_limit, cost = units, 0, 0.0
    for limit, rate in slabs:
        slab_units = min(remaining, limit - prev_limit)
        if slab_units <= 0:
            break
        cost += slab_units * rate
        remaining -= slab_units
        prev_limit = limit
        if remaining <= 0:
            break
    return round(cost, 2)

SEASONAL_FACTORS = {
    "Air Conditioner": {3:1.3, 4:1.7, 5:1.9, 6:1.7, 7:1.3, 8:1.2, 9:1.1, 10:0.9, 11:0.4, 12:0.3, 1:0.3, 2:0.5},
    "Air Cooler":      {3:1.3, 4:1.6, 5:1.8, 6:1.6, 7:1.2, 8:1.1, 9:1.0, 10:0.8, 11:0.4, 12:0.3, 1:0.3, 2:0.5},
    "Space Heater":    {11:1.3, 12:1.8, 1:1.9, 2:1.4, 3:0.6, 10:0.3},
    "Water Heater":    {11:1.2, 12:1.5, 1:1.6, 2:1.3, 3:1.0, 10:0.9},
    "Ceiling Fan":     {4:1.2, 5:1.3, 6:1.3, 7:1.1, 8:1.1},
}

def seasonal_factor(name, month):
    return SEASONAL_FACTORS.get(name, {}).get(month, 1.0)

def daily_kwh(appliance):
    return appliance["power_w"] * appliance["count"] * appliance["hours"] / 1000

def monthly_kwh(appliances, month):
    return round(sum(daily_kwh(a) * 30 * seasonal_factor(a["name"], month) for a in appliances), 2)

def yearly_breakdown(appliances):
    return {m: monthly_kwh(appliances, m) for m in range(1, 13)}

#### Screen 1: House type selection

In [3]:
class HouseSelectScreen(tk.Frame):
    def __init__(self, parent, controller):
        super().__init__(parent)
        self.controller = controller
        tk.Label(self, text="Select Your House Type", font=("Segoe UI", 20, "bold")).pack(pady=40)
        btn_frame = tk.Frame(self)
        btn_frame.pack()
        for house in HOUSE_FILES:
            tk.Button(btn_frame, text=house, width=20, height=2, font=("Segoe UI", 12),
                      command=lambda h=house: self.select(h)).pack(pady=6)

    def select(self, house):
        self.controller.house_type = house
        self.controller.appliances = load_house_data(house)
        self.controller.show_frame(ApplianceEditScreen)

#### Screen 2: Appliance editor (+ Add/Edit dialog)